<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Ffinal-retraining/notebooks/3_4_final_retraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4.8 - Reentrenamiento final: tabla comparativa del Goal 2.4

**Objetivo.** La tabla comparativa de U-Net, SegFormer y UWSegFormer evaluados con **las mismas
etiquetas, las mismas particiones y las mismas metricas**, con la linea base "todo estructura" al
lado y 5 semillas.

**Reparto.** Este notebook entrena **solo el U-Net**: 7 folds x 5 semillas = 35 corridas. Dayana no
entrena para la 4.8, reutiliza las corridas de su task 5.4 bajo el mismo protocolo y exporta sus
`per_frame` a la carpeta compartida de Drive. La seccion 6 monta la tabla con lo que haya.

**Protocolo.** Las ocho decisiones estan en `docs/protocolo_evaluacion.md`. Las que afectan al
codigo de este notebook:

| # | decision | donde se aplica |
|---|---|---|
| 1 | las 68 mascaras tal cual, reportando el peso por video | manifiesto |
| 2 | promedio macro: por imagen, luego por video, luego entre semillas | arnes 5.7 |
| 3 | presupuesto fijo de pasadas, **sin seleccion de checkpoint** | seccion 4 |
| 4 | regla de empate explicita | seccion 6 |
| 5 | leave-one-video-out, 7 folds | manifiesto |
| 6 | semillas [0, 1, 2, 3, 4] | seccion 4 |
| 7 | medicion a la resolucion original del frame | `src/training/predict.py` |
| 8 | solo `evaluate()` del arnes 5.7 | seccion 4 |

---

## Estado: no ejecutable todavia

Faltan tres precondiciones, todas de otras personas:

1. **Las 68 mascaras de la task 5.1.** Hoy hay 28 (Video1 x20, Video6 x8), que dan 2 folds en vez
   de 7.
2. **El manifiesto `folds_4_8.csv` con su sha256**, que genera Anthony con
   `python -m src.data.build_fold_manifest`.
3. **El presupuesto de pasadas**, que sale de la corrida piloto de la seccion 3 y hay que acordar
   con Dayana.

Con lo que hay hoy el notebook corre de punta a punta en modo piloto (`PILOTO = True`) y sirve para
dejar fijado el presupuesto. La corrida de las 35 viene despues.

**Las etiquetas vienen de `Test_Set_Curado`, no de `mmseg_pipes`.** Es deliberado: `mmseg_pipes` se
construye desde `SAM_Masks_Frames_CLAHE`, y la task 4.3 mide esas mascaras contra el ground truth
curado obteniendo IoU 0,083 en Video6, donde marcan el 95 % del frame mientras la estructura real
ocupa el 8 %.

## 1. Entorno

In [ ]:
import hashlib
import json
import math
import os
import random
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/anthony-yt/deepwater-image-segmentation-classical-quantum.git"
BRANCH = "feat/final-retraining"

# El arnes de la task 5.7 vive en otra rama sin mergear.
DEPENDENCIAS = [("feat/labelling-evaluation-tools", "src/evaluation/segmentation_metrics.py")]

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Under_Water_Dataset")
    if not Path("/content/repo/.git").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    for rama, archivo in DEPENDENCIAS:
        subprocess.run(["git", "fetch", "-q", "origin", rama], check=True)
        subprocess.run(["git", "checkout", f"origin/{rama}", "--", archivo], check=True)
        print("traido", archivo, "de", rama)
else:
    DRIVE_ROOT = Path(os.environ.get("DW_DATA_ROOT", "Under_Water_Dataset")).resolve()
sys.path.insert(0, str(Path.cwd()))

for _, archivo in DEPENDENCIAS:
    assert Path(archivo).exists(), f"falta {archivo}: git checkout origin/<rama> -- {archivo}"

PROCESSED = DRIVE_ROOT / "Dataset_Processed"
FRAMES_DIR = PROCESSED / "Frames_CLAHE"
MASKS_DIR = PROCESSED / "Test_Set_Curado"
MANIFEST = PROCESSED / "Seleccion" / "folds_4_8.csv"
# Carpeta compartida: aqui cada modelo deja sus per_frame, los tres se leen desde la seccion 6.
RESULTS_DIR = PROCESSED / "resultados_4_8"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODELO = "U-Net"
IMG_SIZE = 512
BATCH = 4
LR = 1e-4
SEEDS = [0, 1, 2, 3, 4]          # decision 6
PASADAS = None                    # decision 3: lo fija la corrida piloto de la seccion 3
PILOTO = True                     # True: una semilla, un fold, para fijar PASADAS

# sha256 del manifiesto que debe registrar cada corrida. None = se lee del .meta.json y se avisa.
SHA256_ESPERADO = None

print("frames     :", FRAMES_DIR)
print("mascaras   :", MASKS_DIR)
print("manifiesto :", MANIFEST)
print("resultados :", RESULTS_DIR)

## 2. Manifiesto y folds

El manifiesto es lo que hace que las 35 corridas de este notebook y las 70 de la task 5.4 sean
comparables. Que las dos digamos "hacemos leave-one-video-out" no garantiza el mismo reparto; el
hash si.

In [ ]:
def sha256_of(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_manifest(path: Path, expected=None) -> tuple:
    if not path.exists():
        raise SystemExit(
            f"no existe {path}.\n"
            "Lo genera Anthony con:\n"
            "  python -m src.data.build_fold_manifest "
            f"--masks-dir '{MASKS_DIR}' --out '{path}'")
    checksum = sha256_of(path)
    meta_path = path.with_suffix(".meta.json")
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}

    if expected is not None and checksum != expected:
        raise SystemExit(f"el manifiesto cambio.\n  esperado: {expected}\n  real    : {checksum}\n"
                         "Las corridas anteriores no son comparables con este reparto.")
    if expected is None:
        print(f"ATENCION: SHA256_ESPERADO esta en None. Fija SHA256_ESPERADO = '{checksum}'\n"
              "para que una corrida futura falle si el manifiesto cambia.")
    return pd.read_csv(path), checksum, meta


folds_df, MANIFEST_SHA256, MANIFEST_META = load_manifest(MANIFEST, SHA256_ESPERADO)
FOLDS = sorted(folds_df.fold.unique())

print(f"\nsha256 del manifiesto: {MANIFEST_SHA256}")
print(f"{len(FOLDS)} folds: {FOLDS}")
print(folds_df.groupby(["fold", "split"]).size().unstack(fill_value=0).to_string())
if MANIFEST_META:
    print("\npeso de cada video en el conjunto, y su linea base (decision 1):")
    print(pd.DataFrame(MANIFEST_META["por_video"]).T.to_string())

if len(FOLDS) < 7:
    print(f"\nSOLO {len(FOLDS)} FOLDS: la task 5.1 aun no ha completado los 7 videos. "
          "Esta corrida es piloto, no el resultado de la 4.8.")

In [ ]:
def pairs_from_manifest(fold: str, split: str) -> list:
    """Pares (frame, mascara) de un fold. Lee Test_Set_Curado, no mmseg_pipes."""
    ids = folds_df[(folds_df.fold == fold) & (folds_df.split == split)].frame_id
    pairs = []
    for fid in ids:
        mask = MASKS_DIR / f"{fid}_mask.png"
        frame = next((FRAMES_DIR / f"{fid}{e}" for e in (".jpg", ".jpeg", ".png")
                      if (FRAMES_DIR / f"{fid}{e}").exists()), None)
        if frame is None or not mask.exists():
            print(f"  falta {fid}: frame={frame is not None} mascara={mask.exists()}")
            continue
        pairs.append((str(frame), str(mask)))
    return pairs


for fold in FOLDS:
    tr, te = pairs_from_manifest(fold, "train"), pairs_from_manifest(fold, "test")
    print(f"{fold}: train {len(tr)} | test {len(te)}")

## 3. Presupuesto de pasadas (decision 3)

En leave-one-video-out el video excluido **es** el conjunto de prueba, asi que elegir el mejor
checkpoint segun su metrica seria seleccionar sobre el test. Por eso no hay seleccion: se entrena
un numero fijo de pasadas por el conjunto de entrenamiento y se usan los **pesos finales**.

El numero se fija con una corrida piloto y vale para los tres modelos. Como MMSegmentation cuenta
iteraciones y no epocas, la equivalencia es:

```
pasadas = iteraciones x samples_per_gpu / N        (N = imagenes de entrenamiento del fold)
```

La celda siguiente imprime las iteraciones equivalentes por fold, para que Dayana configure su
lado con el mismo presupuesto.

In [ ]:
def iteraciones_equivalentes(pasadas: int, n_train: int, samples_per_gpu: int = 2) -> int:
    return math.ceil(pasadas * n_train / samples_per_gpu)


def tabla_presupuesto(pasadas: int) -> pd.DataFrame:
    filas = []
    for fold in FOLDS:
        n = len(pairs_from_manifest(fold, "train"))
        filas.append({"fold": fold, "n_train": n,
                      "epocas_unet": pasadas,
                      "iteraciones_mmseg_spg2": iteraciones_equivalentes(pasadas, n, 2),
                      "iteraciones_mmseg_spg4": iteraciones_equivalentes(pasadas, n, 4)})
    return pd.DataFrame(filas).set_index("fold")


if PASADAS is None:
    print("PASADAS esta en None: corre el piloto de abajo y fija el valor arriba.\n")
    print("De referencia, las 2000 iteraciones que usa el SegFormer actual equivalen a:")
    for fold in FOLDS:
        n = len(pairs_from_manifest(fold, "train"))
        if n:
            print(f"  {fold}: 2000 it x 2 / {n} = {2000 * 2 / n:.0f} pasadas")
else:
    print(tabla_presupuesto(PASADAS).to_string())

### 3.1 Corrida piloto

Un solo fold y una sola semilla, con el historial completo, para ver a partir de que pasada la
metrica del test deja de cambiar. **Esto no selecciona el checkpoint**: solo informa el valor de
`PASADAS`, que luego es fijo e igual para todos.

In [ ]:
import albumentations as A
import torch
from torch.utils.data import DataLoader

from src.data.pipe_dataset import PipeDataset
from src.evaluation.segmentation_metrics import across_seeds, evaluate
from src.models.unet import build_unet
from src.training.losses import make_criterion
from src.training.loop import baseline_metrics, train_model
from src.training.predict import predict_masks

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| dispositivo", DEVICE)

TRAIN_TF = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
])
EVAL_TF = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def loaders(fold: str):
    tr = DataLoader(PipeDataset(pairs_from_manifest(fold, "train"), TRAIN_TF),
                    batch_size=BATCH, shuffle=True, num_workers=2)
    te = DataLoader(PipeDataset(pairs_from_manifest(fold, "test"), EVAL_TF),
                    batch_size=BATCH, shuffle=False, num_workers=2)
    return tr, te

In [ ]:
if PILOTO:
    fold = FOLDS[0]
    train_dl, test_dl = loaders(fold)
    print(f"piloto en {fold}: train {len(train_dl.dataset)} | test {len(test_dl.dataset)}")
    print("linea base del fold:", json.dumps(baseline_metrics(test_dl), indent=2))

    set_seed(SEEDS[0])
    modelo = build_unet(attention=False, device=DEVICE)
    piloto = train_model(modelo, train_dl, test_dl, make_criterion("ce_dice"), DEVICE,
                         epochs=60, lr=LR, scheduler="cosine", eval_train=False, log_every=5)

    h = pd.DataFrame(piloto["history"])
    base = piloto["baseline"]["todo_estructura_iou"]
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(h.epoch, h.val_iou, label="IoU en el video excluido")
    ax.axhline(base, ls="--", color="#555", label=f"todo estructura ({base:.3f})")
    ax.set_xlabel("pasadas por el conjunto de entrenamiento")
    ax.set_ylabel("IoU")
    ax.set_title(f"Piloto en {fold}. Elegir PASADAS donde la curva se aplana, "
                 "no donde alcanza su maximo")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
    print(f"\nmaximo en la pasada {int(h.val_iou.idxmax()) + 1}, pero NO se elige por el maximo: "
          "eso seria seleccionar sobre el test. Se elige donde la curva deja de subir.")

## 4. Corrida: 7 folds x 5 semillas

Sin seleccion de checkpoint: se usan los pesos finales. Cada `per_frame` se guarda en la carpeta
compartida con el `sha256` del manifiesto dentro, para poder comprobar despues que los tres
modelos corrieron sobre el mismo reparto.

In [ ]:
def ruta_per_frame(modelo: str, fold: str, seed: int) -> Path:
    tag = modelo.lower().replace("-", "").replace(" ", "")
    return RESULTS_DIR / f"per_frame_{tag}_{fold}_seed{seed}.csv"


def correr(pasadas: int, seeds: list, folds: list) -> list:
    hechas = []
    for fold in folds:
        train_dl, test_dl = loaders(fold)
        if not len(test_dl.dataset):
            print(f"{fold}: sin frames de test, lo salto")
            continue
        for seed in seeds:
            destino = ruta_per_frame(MODELO, fold, seed)
            if destino.exists():
                print(f"{fold} semilla {seed}: ya existe, lo salto")
                continue

            print(f"\n{'=' * 70}\n{MODELO} | {fold} | semilla {seed} | {pasadas} pasadas\n"
                  f"{'=' * 70}")
            set_seed(seed)
            modelo = build_unet(attention=False, device=DEVICE)
            train_model(modelo, train_dl, test_dl, make_criterion("ce_dice"), DEVICE,
                        epochs=pasadas, lr=LR, scheduler="cosine", eval_train=False, log_every=20)
            # Pesos finales a proposito: no se selecciona checkpoint (decision 3).

            preds, gts = predict_masks(modelo, pairs_from_manifest(fold, "test"), DEVICE, IMG_SIZE)
            per_frame, per_video = evaluate(preds, gts, n_boot=1000)
            per_frame["modelo"] = MODELO
            per_frame["fold"] = fold
            per_frame["seed"] = seed
            per_frame["pasadas"] = pasadas
            per_frame["manifest_sha256"] = MANIFEST_SHA256
            per_frame.to_csv(destino, index=False)
            hechas.append(destino.name)

            print(per_video[["video", "n_frames", "iou", "base_iou", "iou_minus_base",
                             "boundary_f1"]].round(4).to_string(index=False))
            del modelo
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    return hechas


if PASADAS is None:
    print("fija PASADAS en la seccion 1 antes de correr. Sale del piloto de la seccion 3.1.")
elif PILOTO:
    print("PILOTO = True: corriendo solo el primer fold y la primera semilla.")
    print(correr(PASADAS, SEEDS[:1], FOLDS[:1]))
else:
    print(correr(PASADAS, SEEDS, FOLDS))

## 5. Resultados de Dayana

Sus corridas de la task 5.4 se leen de la misma carpeta. El formato esperado es el mismo
`per_frame` del arnes, con las columnas `modelo`, `fold`, `seed` y `manifest_sha256`.

La celda comprueba que los tres modelos comparten el `sha256` del manifiesto: si no, no entran en
la misma tabla.

In [ ]:
def cargar_todo() -> pd.DataFrame:
    archivos = sorted(RESULTS_DIR.glob("per_frame_*.csv"))
    if not archivos:
        raise SystemExit(f"no hay resultados en {RESULTS_DIR}")
    todo = pd.concat([pd.read_csv(p) for p in archivos], ignore_index=True)
    for col in ["modelo", "fold", "seed", "manifest_sha256"]:
        if col not in todo.columns:
            raise SystemExit(f"a los per_frame les falta la columna {col}")
    return todo


todo = cargar_todo()
print("corridas por modelo:")
print(todo.groupby("modelo").agg(folds=("fold", "nunique"), semillas=("seed", "nunique"),
                                 frames=("frame_id", "size")).to_string())

hashes = todo.groupby("modelo").manifest_sha256.unique()
print("\nsha256 del manifiesto por modelo:")
for modelo, hs in hashes.items():
    print(f"  {modelo}: {list(hs)}")

distintos = set(todo.manifest_sha256.unique())
if len(distintos) > 1:
    print("\nPROBLEMA: hay corridas sobre manifiestos distintos. No son comparables.")
    print("Modelos afectados:", {m: list(h) for m, h in hashes.items()})
else:
    print(f"\ntodos los modelos corrieron sobre el mismo manifiesto ({list(distintos)[0][:16]}...)")

## 6. Tabla final y regla de empate

Para cada modelo, los `per_frame` de los 7 folds de una semilla se concatenan en una sola tabla:
cada video queda evaluado por el modelo que **no** lo vio. Luego `across_seeds` da media y
desviacion entre semillas, por video.

In [ ]:
def tabla_modelo(modelo: str, datos: pd.DataFrame) -> pd.DataFrame:
    sub = datos[datos.modelo == modelo]
    por_semilla = {seed: g for seed, g in sub.groupby("seed")}
    tabla = across_seeds(por_semilla).reset_index()
    tabla.insert(0, "modelo", modelo)
    return tabla


bloques = [tabla_modelo(m, todo) for m in sorted(todo.modelo.unique())]
final = pd.concat(bloques, ignore_index=True)

# La linea base es determinista: no lleva desviacion entre semillas.
base = (todo.groupby("video").base_iou.mean().rename("iou")
        .reset_index().assign(modelo="todo estructura", iou_sd=0.0, n_seeds=1))

cols = [c for c in ["modelo", "video", "n_seeds", "iou", "iou_sd",
                    "boundary_f1", "boundary_f1_sd", "base_iou"] if c in final.columns]
print(pd.concat([final[cols], base], ignore_index=True).round(4).to_string(index=False))
final.to_csv(RESULTS_DIR / "tabla_4_8.csv", index=False)
print("\nguardado en", RESULTS_DIR / "tabla_4_8.csv")

In [ ]:
def empate(media_a: float, sd_a: float, media_b: float, sd_b: float) -> bool:
    """Decision 4: la diferencia solo es real si supera la suma de las dos desviaciones."""
    return abs(media_a - media_b) <= (sd_a or 0) + (sd_b or 0)


medias = final.pivot_table(index="video", columns="modelo", values="iou")
desv = final.pivot_table(index="video", columns="modelo", values="iou_sd")
base_iou = todo.groupby("video").base_iou.mean()

print("--- comparaciones por video (decision 4) ---\n")
modelos = list(medias.columns)
for video in medias.index:
    print(f"{video}   linea base = {base_iou[video]:.4f}")
    for i, a in enumerate(modelos):
        for b in modelos[i + 1:]:
            ma, mb = medias.loc[video, a], medias.loc[video, b]
            sa = desv.loc[video, a] if a in desv.columns else 0.0
            sb = desv.loc[video, b] if b in desv.columns else 0.0
            if empate(ma, sa, mb, sb):
                veredicto = "EMPATE"
            else:
                veredicto = f"gana {a if ma > mb else b}"
            print(f"   {a} {ma:.4f}+-{sa:.4f}  vs  {b} {mb:.4f}+-{sb:.4f}"
                  f"   |diff|={abs(ma - mb):.4f}  umbral={(sa or 0) + (sb or 0):.4f}  -> {veredicto}")
    for m in modelos:
        mm = medias.loc[video, m]
        sm = desv.loc[video, m] if m in desv.columns else 0.0
        contra_base = "EMPATE con la base" if empate(mm, sm, base_iou[video], 0.0) else (
            "supera la base" if mm > base_iou[video] else "POR DEBAJO de la base")
        print(f"   {m} vs todo estructura -> {contra_base}")
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, metrica, sufijo in [(axes[0], "iou", "iou_sd"), (axes[1], "boundary_f1", "boundary_f1_sd")]:
    if metrica not in final.columns:
        continue
    piv = final.pivot_table(index="video", columns="modelo", values=metrica)
    err = final.pivot_table(index="video", columns="modelo", values=sufijo).fillna(0)
    x = np.arange(len(piv.index))
    ancho = 0.8 / max(len(piv.columns), 1)
    for i, modelo in enumerate(piv.columns):
        ax.bar(x + i * ancho, piv[modelo].values, ancho, yerr=err[modelo].values,
               capsize=3, label=modelo)
    if metrica == "iou":
        ax.plot(x + ancho * (len(piv.columns) - 1) / 2, base_iou.reindex(piv.index).values,
                "k_", markersize=24, label="todo estructura")
    ax.set_xticks(x + ancho * (len(piv.columns) - 1) / 2)
    ax.set_xticklabels(piv.index)
    ax.set_title(metrica)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)
fig.suptitle("Task 4.8 - barras de error = desviacion entre semillas", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Conclusion

*Se escribe tras la corrida de las 35, que requiere las 68 mascaras de la task 5.1.*

### Como leerla

Cada celda se lee contra **la linea base de su propio video**, no contra un umbral absoluto. La
base es 0,718 en Video1 y 0,081 en Video6: un IoU de 0,70 es un empate con no hacer nada en Video1
y un resultado muy bueno en Video6.

Un modelo gana a otro en un video solo si la diferencia de medias supera la suma de las dos
desviaciones entre semillas. Si no, es empate, y el empate es un resultado valido: con 68 mascaras
y folds de 48 a 60 imagenes la variacion entre semillas va a ser grande, y los modelos clasicos se
separan de la base por unos 0,03.

### Que responder por escrito

1. Para cada video: que modelos superan la linea base y cuales no.
2. Si hay un modelo que gane de forma consistente en varios videos, o si la tabla es un empate.
3. Si el orden entre modelos cambia segun el video, y en particular si Video6 (la unica tuberia
   real, con el 8 % de estructura) se comporta distinto del resto.
4. Cuanto de la variacion observada es entre semillas y cuanto entre modelos. Si lo primero domina,
   la conclusion del Goal 2.4 es que con estos datos no se puede elegir entre las tres
   arquitecturas, y eso hay que decirlo asi.

### Precondiciones pendientes

Ver la tabla de `docs/protocolo_evaluacion.md`. Resumidas: las 40 mascaras que faltan de la 5.1, el
manifiesto con su hash, el valor de `PASADAS`, las corridas de la 5.4 de Dayana exportadas a
`resultados_4_8/`, y el bootstrap modelo contra modelo en el arnes para desempatar los casos al
limite.